In [1]:
import numpy as np
from qutip import *
import plotly.graph_objects as px

In [2]:
sz = 0.45e-6
sy = 5.6e-6

rng = np.random.default_rng()
xc = rng.normal(0, sz, 1)[0]
yc = rng.normal(0, sy, 1)[0]
zc = rng.normal(5.1e-6, sz, 1)[0]

xt = rng.normal(0, sz, 1)[0]
yt = rng.normal(0, sy, 1)[0]
zt = rng.normal(-5.1e-6, sz, 1)[0]

In [2]:
import pairinteraction as pi

ket_32 = pi.KetAtom(
    "Rb",
    n=79,
    l=2,
    j=2.5,
    m=0.5
)

ket_52 = pi.KetAtom(
    "Rb",
    n=79,
    l=2,
    j=1.5,
    m=0.5
)

basis_atom = pi.BasisAtom(
    "Rb",
    n=(77, 81),
    l=(0, 3)
)

print(basis_atom)
print("Number of states:", basis_atom.number_of_states)

B_values = np.linspace(0, 15e-4, 200)

systems_atom = [
    pi.SystemAtom(basis_atom)
    .set_magnetic_field([0, 0, B], unit="T")
    for B in B_values
]

pi.diagonalize(systems_atom, diagonalizer="eigen")

BasisAtom('Rb', n=(77, 81), l=(0, 3))
Number of states: 160


In [3]:
pair_systems = []
delta_energy = 0.2

for system in systems_atom[0:200]:

    pair_energy = 2 * system.get_corresponding_energy(
        ket_52,
        unit="GHz"
    )

    pair_basis = pi.BasisPair(
        [system, system],
        m=(-2 * ket_32.m, 2 * ket_52.m),
        energy=(pair_energy - delta_energy,
                pair_energy + 0.6),
        energy_unit="GHz",
    )

    pair_system = pi.SystemPair(pair_basis)

    # R → infinity
    pair_system.set_distance(np.inf, unit="micrometer")

    pair_systems.append(pair_system)

c:\Users\aiap2\AppData\Local\Programs\Python\Python314\Lib\site-packages\pairinteraction\system\system_pair.py:117: RuntimeWarning: invalid value encountered in scalar multiply
  distance_vector = [np.sin(np.deg2rad(angle_degree)) * distance, 0, np.cos(np.deg2rad(angle_degree)) * distance]


In [18]:
pi.diagonalize(pair_systems, diagonalizer="eigen")

pair_energies = np.array([
    system.get_eigenenergies(unit="GHz")
    for system in pair_systems
])

x = []
y = []
for i,B in enumerate(B_values[0:200]):
    for e in pair_energies[i]:
        x.append(B)
        y.append(e)

import plotly.graph_objects as go
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="markers",
    opacity=1,
    marker=dict(size=1, color="#3A5D9C")
))
fig.update_layout(
    xaxis_title="Magnetic Field (10-4T)",
    yaxis_title="U (MHz)",
    template = "plotly_dark",
)